# Chapter 7 -- Planning, Decomposition & Artifacts (Solved)

Work through this notebook **after reading** `notes/ch07-planning-artifacts.md`. This chapter builds the three durable artifacts the notes cover -- a `PLAN.md` with checkboxes and an explicit dependency DAG, an executor that works that DAG respecting dependencies, and a reconciler that rewrites the artifact after every step -- against the exact same 12-subtask avatar-upload feature the notes' Section 11 dry-run computed critical path and parallel width for by hand.

Two exercises below have a stub to fill in: **parallel-width detection** (notes Section 7) and **a re-planning trigger** (notes Section 6). Everything is fully offline and deterministic -- no API key needed for either exercise, and no API key needed for the main walkthrough either, since this chapter's core deliverable is about artifacts and scheduling, not about a live model.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## The Shared Task: the 12-Subtask Avatar-Upload DAG

Same feature, same twelve subtasks, same durations and dependencies as notes Section 11's hand-computed dry-run -- so every number this notebook prints can be checked directly against the notes. Each task is a dict: `id`, `description`, `duration` (hours), `deps` (list of task ids that must be `completed` first), and `state` (Section 3's four-state machine: `pending` / `in_progress` / `completed` / `blocked`).

In [ ]:
TASKS = {
    "T1":  {"description": "DB migration for avatar column",                 "duration": 2, "deps": [],               "state": "pending"},
    "T2":  {"description": "Upload API endpoint (accepts file, no storage)",  "duration": 3, "deps": ["T1"],           "state": "pending"},
    "T3":  {"description": "Storage client (S3-style, no resize)",           "duration": 2, "deps": [],               "state": "pending"},
    "T4":  {"description": "Image resize service",                          "duration": 4, "deps": ["T3"],           "state": "pending"},
    "T5":  {"description": "Wire endpoint to storage + resize",             "duration": 2, "deps": ["T2", "T4"],     "state": "pending"},
    "T6":  {"description": "Frontend upload widget",                        "duration": 3, "deps": [],               "state": "pending"},
    "T7":  {"description": "Frontend preview component",                    "duration": 2, "deps": ["T6"],           "state": "pending"},
    "T8":  {"description": "Wire frontend to API",                          "duration": 2, "deps": ["T5", "T7"],     "state": "pending"},
    "T9":  {"description": "Avatar display component (reads storage)",      "duration": 2, "deps": ["T3"],           "state": "pending"},
    "T10": {"description": "Migration script for existing avatars",         "duration": 3, "deps": ["T1", "T4"],     "state": "pending"},
    "T11": {"description": "End-to-end test suite",                         "duration": 3, "deps": ["T8", "T9", "T10"], "state": "pending"},
    "T12": {"description": "Docs update",                                   "duration": 1, "deps": ["T11"],          "state": "pending"},
}

total_work = sum(t["duration"] for t in TASKS.values())
print(f"{len(TASKS)} subtasks defined, {total_work} hours of total work (sequential baseline).")


## The Planner: Critical Path and `PLAN.md` (Given)

`compute_critical_path` implements notes Section 7's recurrence exactly: a node's longest incoming path is the **max** (not sum) over its dependencies' longest paths, plus its own duration -- because a node only waits for its *slowest* predecessor. `write_plan_md` is the reconciler's other half: it renders the current state of every task as a checkbox list plus a plain-text edge list, so the DAG structure and the checklist state live in one reviewable file, exactly as notes Section 1 and Section 3 argued they should.

In [ ]:
def compute_critical_path(tasks):
    """
    Longest-path-by-max-not-sum recurrence (notes Section 7 / 11).
    Returns (longest_finish, critical_path_hours) where longest_finish is a
    dict of task_id -> earliest possible finish time given unlimited workers.
    """
    longest_finish = {}

    def finish_time(task_id):
        if task_id in longest_finish:
            return longest_finish[task_id]
        task = tasks[task_id]
        start = max((finish_time(dep) for dep in task["deps"]), default=0)
        result = start + task["duration"]
        longest_finish[task_id] = result
        return result

    for task_id in tasks:
        finish_time(task_id)

    critical_path_hours = max(longest_finish.values())
    return longest_finish, critical_path_hours


longest_finish, critical_path_hours = compute_critical_path(TASKS)
print("Longest finish time reaching each node (hours from start):")
for task_id, finish in sorted(longest_finish.items(), key=lambda kv: kv[1]):
    print(f"  {task_id}: {finish}h")
print(f"\nCritical path length: {critical_path_hours} hours")
assert critical_path_hours == 14, f"expected 14h to match the notes' hand-computed critical path, got {critical_path_hours}"
print("Matches notes Section 11's hand-computed value exactly.")


In [ ]:
def write_plan_md(tasks, path="PLAN.md"):
    """
    Render the current checklist + DAG edge list to a markdown file.
    This IS the reconciler -- called after every state change, never batched.
    """
    lines = ["# PLAN: Avatar Upload Feature", ""]
    lines.append("## Checklist")
    checkbox = {"completed": "[x]", "in_progress": "[~]", "blocked": "[!]", "pending": "[ ]"}
    for task_id, task in tasks.items():
        box = checkbox[task["state"]]
        lines.append(f"- {box} **{task_id}** ({task['duration']}h) -- {task['description']}  <!-- state={task['state']} -->")
    lines.append("")
    lines.append("## Dependency Edges")
    for task_id, task in tasks.items():
        if task["deps"]:
            lines.append(f"- {', '.join(task['deps'])} -> {task_id}")
    lines.append("")
    content = "\n".join(lines)
    with open(path, "w") as f:
        f.write(content)
    return content


plan_text = write_plan_md(TASKS)
print(plan_text[:600])
print("...")
print(f"\nWrote PLAN.md ({len(plan_text)} characters) with all 12 tasks pending.")


## The Executor: Working the DAG With a Reconciler (Given)

`run_executor` is this chapter's core deliverable: it repeatedly finds every task whose dependencies are all `completed` and which is still `pending`, "executes" it (here, a deterministic stand-in for a real subagent call -- the point of this notebook is the scheduling and artifact logic, not another scripted model), marks it `completed` only after a stand-in verifier passes, and calls `write_plan_md` again immediately -- printing a **diff** of what changed in the file, one line per step, exactly as the notes' code-deliverable description asks for. Note how many tasks come back `READY` at once in the early steps -- that count is Section 7's parallel width, made visible during a real run instead of only computed by hand.

In [ ]:
def _verify_task(task_id):
    """Stand-in verifier -- every task in this deterministic walkthrough passes on its first real attempt."""
    return True


def _ready_tasks(tasks):
    """Every task that is still pending and whose dependencies are all completed."""
    return [
        task_id for task_id, task in tasks.items()
        if task["state"] == "pending" and all(tasks[dep]["state"] == "completed" for dep in task["deps"])
    ]


def run_executor(tasks, plan_path="PLAN.md"):
    """
    Repeatedly: find every ready task, execute + verify each, mark completed,
    reconcile PLAN.md, and print a diff of exactly what changed.
    """
    previous_text = write_plan_md(tasks, plan_path)
    round_num = 0

    while any(t["state"] != "completed" for t in tasks.values()):
        round_num += 1
        ready = _ready_tasks(tasks)
        if not ready:
            print(f"round {round_num}: no ready tasks and not all completed -- stuck (should not happen on an acyclic DAG)")
            break

        print(f"round {round_num}: {len(ready)} task(s) ready in parallel -> {ready}")
        for task_id in ready:
            tasks[task_id]["state"] = "in_progress"

        for task_id in ready:
            ok = _verify_task(task_id)
            tasks[task_id]["state"] = "completed" if ok else "blocked"
            new_text = write_plan_md(tasks, plan_path)
            changed_line = next(
                line for line in new_text.splitlines()
                if task_id in line and "Checklist" not in line and "state=" in line
            )
            print(f"    reconciled: {changed_line.strip()}")
            previous_text = new_text

    print(f"\nExecutor finished after {round_num} rounds. All tasks: "
          f"{ {tid: t['state'] for tid, t in tasks.items()} }")


run_executor(TASKS)


In [ ]:
completed_count = sum(1 for t in TASKS.values() if t["state"] == "completed")
assert completed_count == 12, f"expected all 12 tasks completed, got {completed_count}"
print(f"Confirmed: all {completed_count} tasks reached 'completed' via the reconciler loop above,")
print("and PLAN.md on disk reflects that final state -- read it back to check for yourself:")
print()
with open("PLAN.md") as f:
    print(f.read())


## Exercise 1 -- Parallel-Width Detection

Implement `compute_parallel_width(tasks)`: notes Section 7 / 11's instant-by-instant simulation, but as a reusable function instead of a hand trace. Simulate the schedule with **unlimited workers** honoring dependencies and durations (every ready task starts the instant its dependencies clear), track how many tasks are running at any given moment, and return `(max_width, finish_time)` -- the largest number of tasks ever running simultaneously, and the total time the full schedule takes (which should equal the critical path). Use a simple discrete-event approach: process tasks in order of their earliest possible start time, and track `(start, end)` intervals; the width at any instant is the count of intervals covering that instant.

In [ ]:
def compute_parallel_width(tasks):
    """
    Simulate unlimited-worker scheduling and return (max_width, finish_time).
    Uses the same earliest-start recurrence as compute_critical_path, then
    counts how many [start, end) intervals overlap at any point in time.
    """
    starts, ends = {}, {}

    def compute_times(task_id):
        if task_id in ends:
            return
        task = tasks[task_id]
        for dep in task["deps"]:
            compute_times(dep)
        start = max((ends[dep] for dep in task["deps"]), default=0)
        starts[task_id] = start
        ends[task_id] = start + task["duration"]

    for task_id in tasks:
        compute_times(task_id)

    finish_time = max(ends.values())

    # Sweep every distinct start/end timestamp and count overlapping intervals.
    checkpoints = sorted(set(starts.values()) | set(ends.values()))
    max_width = 0
    for t in checkpoints:
        width = sum(1 for task_id in tasks if starts[task_id] <= t < ends[task_id])
        max_width = max(max_width, width)

    return max_width, finish_time


In [ ]:
# Recompute against a FRESH copy of TASKS (the executor above mutated states,
# but compute_parallel_width only reads durations/deps, not state, so this
# also demonstrates the function is state-independent -- pure schedule math).
max_width, finish_time = compute_parallel_width(TASKS)

print(f"Computed max parallel width: {max_width}")
print(f"Computed schedule finish time: {finish_time}h")

assert max_width == 4, f"expected max parallel width of 4 to match notes Section 11, got {max_width}"
assert finish_time == 14, f"expected finish time of 14h to match the critical path, got {finish_time}"
print("\nBoth match notes Section 11's hand-traced values exactly: width 4, achieved")
print("during the T2/T4/T9/T6 overlap, and a 14h floor no amount of extra workers can beat.")


## Exercise 2 -- A Re-Planning Trigger

Implement `check_replan_trigger(tasks, task_id, new_info)`: notes Section 6's discipline that a re-plan must **cite a specific trigger and a specific new fact**, never fire on a vague feeling. Given a task that just failed verification repeatedly, and a `new_info` string describing what was learned, return a dict `{"trigger": ..., "task_id": ..., "reason": ..., "action": ...}` where `trigger` is one of the four named triggers from the notes (`"contradicted_assumption"`, `"repeated_verifier_failure"`, `"blocked_dependency"`, `"discovered_scope"`), chosen by inspecting `new_info` for the keywords below, and `action` is `"mark_blocked_and_replan"` if a trigger is found, or `"no_replan_needed"` if `new_info` doesn't match any trigger (guarding against replanning on nothing, per the notes).

In [ ]:
def check_replan_trigger(tasks, task_id, new_info):
    """
    Classify new_info into one of the four Section 6 triggers by keyword match,
    or report no_replan_needed if nothing genuinely justifies tearing up the plan.
    """
    lowered = new_info.lower()
    trigger = None
    if "assum" in lowered or "schema" in lowered:
        trigger = "contradicted_assumption"
    elif "failed" in lowered and ("again" in lowered or "repeat" in lowered or "still" in lowered):
        trigger = "repeated_verifier_failure"
    elif "block" in lowered or "unavailable" in lowered:
        trigger = "blocked_dependency"
    elif "scope" in lowered or "also needs" in lowered or "discovered" in lowered:
        trigger = "discovered_scope"

    if trigger is None:
        return {"trigger": None, "task_id": task_id, "reason": new_info, "action": "no_replan_needed"}

    tasks[task_id]["state"] = "blocked"
    return {"trigger": trigger, "task_id": task_id, "reason": new_info, "action": "mark_blocked_and_replan"}


In [ ]:
import copy

sample_tasks = copy.deepcopy(TASKS)

case_a = check_replan_trigger(sample_tasks, "T4", "the storage client's schema doesn't support the resize assumption we planned around")
case_b = check_replan_trigger(sample_tasks, "T11", "the test suite failed again, same failure as the last two attempts")
case_c = check_replan_trigger(sample_tasks, "T10", "the migration script is blocked -- the legacy avatar table is unavailable in staging")
case_d = check_replan_trigger(sample_tasks, "T9", "we discovered this feature also needs admin-only avatar moderation, outside original scope")
case_e = check_replan_trigger(sample_tasks, "T6", "the frontend widget looks fine to me")

assert case_a["trigger"] == "contradicted_assumption"
assert case_b["trigger"] == "repeated_verifier_failure"
assert case_c["trigger"] == "blocked_dependency"
assert case_d["trigger"] == "discovered_scope"
assert case_e["action"] == "no_replan_needed", "a vague, non-triggering observation must NOT force a re-plan"

for case in (case_a, case_b, case_c, case_d, case_e):
    print(f"  task={case['task_id']:4s} trigger={str(case['trigger']):26s} action={case['action']}")

print("\nExercise 2 PASSED -- four genuine triggers are correctly classified and cited,")
print("and a vague, non-triggering observation is correctly rejected as grounds for a")
print("re-plan -- exactly the plan-thrash guard notes Section 6 argues for.")


## Optional -- Have a Real Claude Model Draft the Specify-Phase Milestone Artifact

Notes Section 4 and Section 9: before any decomposition happens, a milestone artifact (objective, boundaries, acceptance criteria) should exist and be reviewed. This optional cell asks a real Claude model, via `AnthropicBedrockMantle`, to draft that artifact for the same avatar-upload feature -- a genuinely live call, not scripted, so whatever the model actually proposes is what gets printed.

In [ ]:
RUN_REAL_SPEC_DEMO = False


def run_real_spec_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real spec demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    prompt = (
        "Draft a short Specify-phase milestone artifact (objective, boundaries, "
        "and 4-6 mechanically checkable acceptance criteria) for a feature that "
        "lets users upload and resize a profile avatar. Keep it under 200 words."
    )
    try:
        response = real_client.messages.create(
            model=MODEL_NAME, max_tokens=500,
            messages=[{"role": "user", "content": prompt}],
        )
        text = next((b.text for b in response.content if b.type == "text"), "")
        print(text)
    except Exception as exc:
        print(f"Real spec demo failed: {type(exc).__name__}: {exc}")


if RUN_REAL_SPEC_DEMO:
    run_real_spec_demo()
else:
    print("RUN_REAL_SPEC_DEMO is False -- running in offline mode only.")
    print("Flip it to True to have a real Claude model draft the milestone artifact via Bedrock.")


## Key Takeaways

You built all three of this chapter's durable artifacts against one real DAG: a `PLAN.md` reconciler that never batches its writes, an executor that respects real dependency structure instead of a flat list, and a critical-path calculation that matched the notes' hand-computed 14-hour floor exactly. The parallel-width function confirmed, in code, that this specific decomposition can never usefully occupy more than 4 workers at once -- a fact about the DAG's shape, not about how well anything schedules around it. The re-planning classifier enforced the one discipline notes Section 6 insists on: a re-plan must name a real trigger and cite a real new fact, or it doesn't happen at all.

**Connection forward:** Chapter 8 returns to a problem this chapter's plan made sharper without solving -- a 12-subtask plan drawing from a hundred available tools means every subtask's context is now competing with schemas for capabilities it may never touch. Skills and progressive disclosure scale *capability count* the same way this chapter scaled *goal size*: by refusing to hold everything in context at once.